# RAG permutation study - GPU runs on Colab

Runs the GPU half of the study on a Colab T4. The local RTX 3060 is retired for
generation, so the whole study sits on one GPU and one numerical regime.

**This notebook is deliberately thin.** All orchestration lives in `src/run.py`;
the cells below do environment setup and shell out to the existing CLI. Per the
repo rule, no analysis logic goes in a notebook - if you find yourself computing a
metric here, it belongs in `src/metrics.py` or `src/stats.py` instead.

**Run the cells in order.** Sections 6 and 7 are gates: if either fails, stop and
read what it says before spending twelve hours on the main grid.

### Sessions will die

Free Colab gives ~12h at most, disconnects after ~90 min idle, and can reclaim the
instance at any time. The main run is ~42,300 generations (~9-12h), so expect 3-5
sittings. That is survivable **only** because the SQLite cache makes re-running the
identical command resume where the last session stopped. Section 5 restores it and
section 9 syncs it back - do not skip either.

One consequence to expect rather than debug: a session that dies mid-run writes
**no `generations.csv`**. Everything paid for is safe in the cache; the CSV appears
on whichever run finally completes.


## 1 - GPU check


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    'No GPU. Runtime > Change runtime type > T4 GPU, then rerun this cell. '
    'A 3B model on CPU makes the main run take days rather than hours.'
)
props = torch.cuda.get_device_properties(0)
print(f'{props.name}, {props.total_memory / 1e9:.1f} GB')
print('torch', torch.__version__)


## 2 - Mount Drive

Drive holds the one irreplaceable artifact: the generation cache. Everything else
- code, weights, results - can be rebuilt.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/rag-permutation-study'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive workspace:', DRIVE_DIR)


## 3 - Clone the repo

The repo is private. Put a GitHub personal access token (classic, `repo` scope) in
Colab's **Secrets** panel - the key icon in the left sidebar - named `GH_TOKEN`,
with notebook access enabled. It is read via `userdata` so the token never appears
in this file.

The remote is rewritten without the token afterwards, so it is not left sitting in
`.git/config`.


In [ ]:
import subprocess
from google.colab import userdata

REPO = 'Rahulrayy/rag-permutation-study'
WORK = '/content/rag'
CLEAN_URL = f'https://github.com/{REPO}.git'


def run(cmd, redact=None):
    """Run a command, echoing output with `redact` masked.

    Output is captured rather than streamed so a failed clone cannot print the
    token back into the notebook in its error message.
    """
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if redact:
        out = out.replace(redact, '***')
    if out:
        print(out)
    if r.returncode:
        raise RuntimeError(f'{cmd[0]} failed with exit code {r.returncode}')


token = userdata.get('GH_TOKEN')
AUTH_URL = f'https://{token}@github.com/{REPO}.git'

if not os.path.exists(WORK):
    run(['git', 'clone', AUTH_URL, WORK], redact=token)
    # Drop the token from the stored remote so it is not left in .git/config.
    run(['git', '-C', WORK, 'remote', 'set-url', 'origin', CLEAN_URL])
else:
    # Supply the token inline rather than relying on the stored remote, which
    # is deliberately tokenless. A bare `git pull` against a private repo has
    # no credentials and dies with 'could not read Username' -- which is what
    # re-running this cell used to do.
    run(['git', '-C', WORK, 'pull', AUTH_URL, 'master'], redact=token)

del token, AUTH_URL

%cd {WORK}
run(['git', '-C', WORK, 'log', '--oneline', '-3'])
run(['git', '-C', WORK, 'remote', '-v'])   # confirm: no token in the remote


## 4 - Dependencies

Three things worth knowing, each of which otherwise costs you a session:

- **torch is skipped.** Colab ships a torch matched to its own CUDA runtime.
  `requirements.txt` says torch must come from the cu128 index, which is right on
  the Windows machine and wrong here - reinstalling risks breaking the runtime for
  no gain.
- **`requirements.txt`, not `requirements.lock`.** Its own header says the loose
  ranges are 'for a fresh setup' and the lock is 'for an exact reproduction'. The
  lock's pins would cascade upgrades through torch's dependency chain.
- **nltk data is required, not optional.** `provence_rerank` and `provence_full`
  load remote code that imports nltk and refuses to start without punkt. A fresh
  Colab session has none, and the failure lands in the selection phase - after GPU
  time is already spent.


In [ ]:
# Everything except torch.
!grep -v '^torch' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

# Provence's remote code will not load without these.
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import transformers, torch
print('transformers', transformers.__version__)
print('torch       ', torch.__version__)
# src/generate.py passes `dtype=` to from_pretrained, which is the 5.x name.
assert int(transformers.__version__.split('.')[0]) >= 5, (
    'transformers 5.x required: generate.py uses the `dtype=` kwarg, which 4.x '
    'calls `torch_dtype`. Run: pip install -q -U "transformers>=5.0"'
)


## 5 - Restore the cache from Drive

`git clone` is not enough. Only ~50 files are tracked and `.gitignore` excludes
`cache/`, so the clone arrives with no generation history at all.

The cache runs on **local Colab disk**, never on the Drive mount - SQLite's file
locking and WAL do not behave over Drive's FUSE layer. Drive holds snapshots.


In [ ]:
import shutil, sqlite3, os

LOCAL_CACHE = '/content/rag/cache/generations.sqlite'
DRIVE_CACHE = os.path.join(DRIVE_DIR, 'generations.sqlite')
os.makedirs(os.path.dirname(LOCAL_CACHE), exist_ok=True)

if os.path.exists(DRIVE_CACHE):
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    n = sqlite3.connect(LOCAL_CACHE).execute(
        'SELECT COUNT(*) FROM generations').fetchone()[0]
    print(f'restored {n} cached generations from Drive')
else:
    print('no cache on Drive yet - starting cold, which is fine')
    print('(to seed it, copy cache/generations.sqlite from the laptop into')
    print(f' {DRIVE_DIR} - checkpoint its WAL first, see section 9)')


### Cache sync helper

Uses SQLite's online **backup API**, not a file copy. Two reasons: the backup API
is built to snapshot a database that is being written to, and a plain copy of the
`.sqlite` silently loses whatever is still in the `-wal` file - which is routinely
the larger of the two.

The snapshot is written to local disk first, then copied to Drive as an ordinary
static file. No live SQLite operation ever touches the FUSE mount.


In [ ]:
import threading, time

def sync_cache_to_drive(verbose=True):
    """Snapshot the live cache and copy it to Drive. Safe to call mid-run."""
    if not os.path.exists(LOCAL_CACHE):
        return 0
    tmp = '/content/cache_snapshot.sqlite'
    src = sqlite3.connect(LOCAL_CACHE)
    dst = sqlite3.connect(tmp)
    with dst:
        src.backup(dst)          # consistent snapshot, WAL included
    n = dst.execute('SELECT COUNT(*) FROM generations').fetchone()[0]
    dst.close()
    src.close()
    shutil.copy(tmp, DRIVE_CACHE)
    os.remove(tmp)
    if verbose:
        print(f'[sync] {n} generations -> Drive')
    return n


_stop = threading.Event()

def _loop(every_s):
    while not _stop.wait(every_s):
        try:
            sync_cache_to_drive(verbose=False)
        except Exception as exc:            # never kill the run over a sync
            print('[sync] failed, continuing:', exc)


def start_autosync(every_s=300):
    """Background snapshot every 5 min. At ~0.8 s/gen that caps loss at ~380."""
    _stop.clear()
    threading.Thread(target=_loop, args=(every_s,), daemon=True).start()
    print(f'autosync every {every_s}s')


start_autosync()
sync_cache_to_drive()


## 6 - Gate 1: tests, then smoke

`pytest` catches Linux/path breakage in seconds. `src.smoke` is the real gate:

**Check 3b generates a batch with the probe first versus last and asserts the
outputs match.** That is precisely the risk in raising `batch_size` on a 16 GB
card - padding changes reduction order in the matmuls, and greedy output can move
with it. `configs/main_colab.yaml` deliberately leaves `batch_size: 4` until this
has been checked at whatever value you intend to use.


In [ ]:
!python -m pytest -q


In [ ]:
!python -m src.smoke


## 7 - Gate 2: cross-device determinism

`src/cache.py` keys generations on `sha256(model, prompt, decode_params)` -
**no GPU, no batch size, no dtype**. Rows produced on the laptop 3060 and rows
produced on this T4 collide under one identical key, with nothing recording which
machine made them.

That is not hypothetical. The cache holds ~1,049 `Qwen/Qwen2.5-3B-Instruct` rows
from the week-1 pilot; pilot and main share model, quantization, seed and
permutation strategies; and `full` ignores budget, so its prompts at budgets 2/3/5
are byte-identical to the pilot's at budget 10. **Those rows will be reused by the
main run** unless this gate says they may be.

`--audit` re-issues cached prompts on this GPU, bypassing the cache wrapper, and
reports the byte-identical rate.

| result | action |
|---|---|
| all identical | Pilot rows are sound. Reuse them; record the rate in ANALYSIS_PLAN section 9. |
| any divergence | Run the next cell to drop the 3060 rows and regenerate here (~15 min). Record the split and the reason in section 9. |


In [ ]:
!python -m src.run --config configs/pilot.yaml --audit 50


In [ ]:
# ONLY run this if the audit above reported a divergence.
# It deletes every generation made on the old GPU so the main run regenerates
# them here, keeping the whole study in one numerical regime.

DROP_LOCAL_GPU_ROWS = False   # flip to True only on a failed audit

if DROP_LOCAL_GPU_ROWS:
    con = sqlite3.connect(LOCAL_CACHE)
    with con:
        n = con.execute(
            "DELETE FROM generations WHERE model = 'Qwen/Qwen2.5-3B-Instruct'"
        ).rowcount
    con.close()
    print(f'dropped {n} rows generated on the old GPU')
    sync_cache_to_drive()
else:
    print('skipped - audit passed, or has not been run yet')


## 8 - The runs, cheapest first

Each stage is a gate. Do not jump to the twelve-hour job.

### 8a - All-arms pilot, n=100

This is week 3's outstanding item ('OAE and RFR computable end to end') **and** the
T4 validation. It exercises every arm - Provence, LLMLingua-2, the LLM pruner, the
oracle - at roughly a tenth of the main run's cost. If an arm is going to fail on
Colab, it fails here in minutes rather than hours.


In [ ]:
!python -m src.run --config configs/main_colab.yaml --n 100
sync_cache_to_drive()


### 8b - Main run, n=300

~42,300 generations, ~9-12h, so **expect to run this cell several times across
several sessions**. Re-running the identical command is the resume mechanism:
`run()` rebuilds the grid each invocation and the cache resolves hits before
generating, so each pass picks up where the last stopped.

Watch for the `arm: selected i/N cells` lines during the selection phase - that
phase does not go through the batch reporting, so those lines are how you tell a
slow arm from a hung one.


In [ ]:
!python -m src.run --config configs/main_colab.yaml
sync_cache_to_drive()


## 9 - Save everything back to Drive

Run this **before** closing the session, and any time you are about to lose the
runtime. The autosync thread only lives as long as the kernel.


In [ ]:
sync_cache_to_drive()

# Results are small; copy the whole tree.
drive_results = os.path.join(DRIVE_DIR, 'results')
if os.path.exists('/content/rag/results'):
    shutil.copytree('/content/rag/results', drive_results, dirs_exist_ok=True)
    print('results ->', drive_results)

!ls -la /content/rag/results/*/ 2>/dev/null | head -40


## 10 - End-to-end proof

The check that says the environment is sound: re-run the gate on the pilot and
confirm it reproduces `results/pilot_w1/gate_report.txt` - **median within-query SD
0.0263 over 100 queries**.

Exact reproduction means this machine agrees with the one that produced the
registered week-1 result. If it does not reproduce, stop and find out why before
trusting anything from the main grid.


In [ ]:
!python -m src.gate results/pilot_w1/generations.csv
